# Google Routes API Walkthrough

This notebook walks through the **Google Routes API v2** (`computeRoutes`) step by step.
By the end, you'll know how to:

1. Get distance & duration between two places
2. Get an **encoded polyline** (the line you draw on a map)
3. Decode the polyline into lat/lng coordinates
4. Handle **multiple stops** (intermediates/waypoints)
5. Use different **travel modes** (drive, walk, bike, transit)

**API Reference:** https://developers.google.com/maps/documentation/routes/compute_route_directions

## 0. Setup

In [2]:
import requests
import os
import json
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
ROUTES_URL = "https://routes.googleapis.com/directions/v2:computeRoutes"

print("API key loaded:", "Yes" if API_KEY else "NO - check your .env file!")

API key loaded: Yes


## 1. Basic Route: Distance & Duration

The simplest `computeRoutes` call. We just need:
- `origin` and `destination` (can be addresses or lat/lng)
- `travelMode` (DRIVE, WALK, BICYCLE, TRANSIT, TWO_WHEELER)
- A **field mask** header telling Google which fields to return

The field mask is key — Google only returns the fields you ask for. This keeps responses fast and billing low.

In [3]:
# --- Basic request: distance and duration between two addresses ---

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "routes.distanceMeters,routes.duration"
}

body = {
    "origin": {
        "address": "University of California, Irvine"
    },
    "destination": {
        "address": "Disneyland, Anaheim, CA"
    },
    "travelMode": "DRIVE"
}

response = requests.post(ROUTES_URL, json=body, headers=headers)
data = response.json()

route = data["routes"][0]
distance_km = route["distanceMeters"] / 1000
duration_min = int(route["duration"].rstrip("s")) / 60

print(f"Distance: {distance_km:.1f} km")
print(f"Duration: {duration_min:.0f} minutes")
print(f"\nRaw response: {json.dumps(data, indent=2)}")

Distance: 27.4 km
Duration: 21 minutes

Raw response: {
  "routes": [
    {
      "distanceMeters": 27372,
      "duration": "1243s"
    }
  ]
}


### Key takeaways
- `duration` comes back as a string like `"626s"` (seconds). Parse it with `.rstrip("s")`.
- `distanceMeters` is an integer. Divide by 1000 for km, or 1609 for miles.
- You can also use lat/lng instead of addresses:
  ```python
  "origin": { "location": { "latLng": { "latitude": 33.6461, "longitude": -117.8427 } } }
  ```

## 2. Getting the Polyline (the line you draw on a map)

A **polyline** is an encoded string that represents a series of lat/lng points along the route.
You decode it to get coordinates, then draw it on a map.

To get it, add `routes.polyline.encodedPolyline` to your field mask.

In [4]:
# --- Request with polyline ---

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "routes.distanceMeters,routes.duration,routes.polyline.encodedPolyline"
}

body = {
    "origin": {
        "address": "University of California, Irvine"
    },
    "destination": {
        "address": "Disneyland, Anaheim, CA"
    },
    "travelMode": "DRIVE"
}

response = requests.post(ROUTES_URL, json=body, headers=headers)
data = response.json()

encoded_polyline = data["routes"][0]["polyline"]["encodedPolyline"]
print(f"Encoded polyline (first 100 chars): {encoded_polyline[:100]}...")
print(f"Full length: {len(encoded_polyline)} characters")

Encoded polyline (first 100 chars): w}hlEfqinUy@yAaAoAwGoGOTt@t@XTj@h@`EtE|@xAt@vAMXcBhB_HfIaBzAkBtAgB~@wAn@MVsEhA{GfAwFv@oD`@gXvDmCj@uB...
Full length: 679 characters


## 3. Decoding the Polyline

The encoded polyline is a compressed string. We need to decode it into `[lat, lng]` pairs.

Google's encoding algorithm is documented here: https://developers.google.com/maps/documentation/utilities/polylinealgorithm

We can use the `polyline` library, or decode it ourselves. Let's do both.

In [5]:
# --- Method 1: Manual decoder (no extra dependency) ---

def decode_polyline(encoded: str) -> list[tuple[float, float]]:
    """Decode a Google encoded polyline into a list of (lat, lng) tuples."""
    points = []
    index = 0
    lat = 0
    lng = 0

    while index < len(encoded):
        # Decode latitude
        shift = 0
        result = 0
        while True:
            b = ord(encoded[index]) - 63
            index += 1
            result |= (b & 0x1F) << shift
            shift += 5
            if b < 0x20:
                break
        dlat = ~(result >> 1) if (result & 1) else (result >> 1)
        lat += dlat

        # Decode longitude
        shift = 0
        result = 0
        while True:
            b = ord(encoded[index]) - 63
            index += 1
            result |= (b & 0x1F) << shift
            shift += 5
            if b < 0x20:
                break
        dlng = ~(result >> 1) if (result & 1) else (result >> 1)
        lng += dlng

        points.append((lat / 1e5, lng / 1e5))

    return points


coords = decode_polyline(encoded_polyline)
print(f"Decoded {len(coords)} coordinate points")
print(f"\nFirst 5 points (lat, lng):")
for lat, lng in coords[:5]:
    print(f"  ({lat:.5f}, {lng:.5f})")
print(f"\nLast point (destination): ({coords[-1][0]:.5f}, {coords[-1][1]:.5f})")

Decoded 182 coordinate points

First 5 points (lat, lng):
  (33.63820, -117.85508)
  (33.63849, -117.85463)
  (33.63882, -117.85423)
  (33.64022, -117.85287)
  (33.64030, -117.85298)

Last point (destination): (33.81609, -117.92252)


### What you'd do on the frontend

On the React Native side, you DON'T need to decode manually.
- **`react-native-maps`** has a `Polyline` component that accepts decoded coordinates.
- The `@mapbox/polyline` npm package can decode for you:
  ```js
  import polyline from '@mapbox/polyline';
  const coords = polyline.decode(encodedPolyline);
  // returns [[lat, lng], [lat, lng], ...]
  ```

## 4. Multiple Stops (Intermediates / Waypoints)

For an itinerary with multiple places, you can add **intermediates** (waypoints) between origin and destination.
The API will compute a route that visits them in order.

This is what you'll use when a user has selected 3-5 places for their itinerary.

In [6]:
# --- Multi-stop route: UCI -> Boba -> Lunch -> Disneyland ---

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": (
        "routes.distanceMeters,"
        "routes.duration,"
        "routes.polyline.encodedPolyline,"
        "routes.legs.distanceMeters,"
        "routes.legs.duration,"
        "routes.legs.polyline.encodedPolyline"
    )
}

body = {
    "origin": {
        "address": "University of California, Irvine"
    },
    "destination": {
        "address": "Disneyland, Anaheim, CA"
    },
    "intermediates": [
        { "address": "TP TEA, Irvine, CA" },
        { "address": "Din Tai Fung, Costa Mesa, CA" }
    ],
    "travelMode": "DRIVE"
}

response = requests.post(ROUTES_URL, json=body, headers=headers)
data = response.json()

route = data["routes"][0]
print(f"Total distance: {route['distanceMeters'] / 1000:.1f} km")
print(f"Total duration: {int(route['duration'].rstrip('s')) / 60:.0f} min")
print(f"\n--- Legs (segments between stops) ---")

stops = ["UCI", "TP TEA", "Din Tai Fung", "Disneyland"]
for i, leg in enumerate(route["legs"]):
    dist = leg["distanceMeters"] / 1000
    dur = int(leg["duration"].rstrip("s")) / 60
    has_polyline = "polyline" in leg
    print(f"  {stops[i]} -> {stops[i+1]}: {dist:.1f} km, {dur:.0f} min (polyline: {has_polyline})")

Total distance: 34.7 km
Total duration: 43 min

--- Legs (segments between stops) ---
  UCI -> TP TEA: 6.0 km, 12 min (polyline: True)
  TP TEA -> Din Tai Fung: 6.1 km, 11 min (polyline: True)
  Din Tai Fung -> Disneyland: 22.5 km, 20 min (polyline: True)


### Understanding legs vs route

| Level | What it is | Use case |
|-------|-----------|----------|
| `routes[0]` | The entire trip (all stops combined) | Total time, overall polyline |
| `routes[0].legs[i]` | One segment (stop A -> stop B) | Travel time between each stop |
| `routes[0].legs[i].steps[j]` | Turn-by-turn directions | "Turn left on Main St" |

For an itinerary map, you'll want **both**:
- The **route-level polyline** to draw the full route line
- The **leg-level durations** to show travel time between each stop

## 5. Travel Modes

Available `travelMode` values:

| Mode | Description |
|------|-------------|
| `DRIVE` | Car routing (default) |
| `WALK` | Walking directions |
| `BICYCLE` | Bike routing |
| `TRANSIT` | Public transit (bus, train, etc.) |
| `TWO_WHEELER` | Motorcycle/scooter routing |

Your app already stores transport preferences from onboarding step 3!
Map those to these values.

In [7]:
# --- Compare travel modes ---

modes = ["DRIVE", "WALK", "BICYCLE", "TRANSIT"]

for mode in modes:
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "routes.distanceMeters,routes.duration"
    }
    body = {
        "origin": { "address": "University of California, Irvine" },
        "destination": { "address": "South Coast Plaza, Costa Mesa, CA" },
        "travelMode": mode
    }
    resp = requests.post(ROUTES_URL, json=body, headers=headers)
    result = resp.json()
    
    if "routes" in result and result["routes"]:
        r = result["routes"][0]
        dist = r["distanceMeters"] / 1000
        dur = int(r["duration"].rstrip("s")) / 60
        print(f"{mode:10s} -> {dist:.1f} km, {dur:.0f} min")
    else:
        print(f"{mode:10s} -> Not available (may not be supported for this route)")

DRIVE      -> 8.4 km, 8 min
WALK       -> 8.5 km, 119 min
BICYCLE    -> 9.9 km, 35 min
TRANSIT    -> 14.7 km, 54 min


## 6. Using Lat/Lng (what you'll get from Places API)

Your backend currently uses addresses from the Places API. But for the map, you'll also want **coordinates**.

The Places API can return `places.location` which gives you `{ latitude, longitude }`.
You can pass these directly to the Routes API.

In [8]:
# --- First, get coordinates from Places API ---

places_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location"
}

resp = requests.post(
    "https://places.googleapis.com/v1/places:searchText",
    json={"textQuery": "boba spots near UC Irvine"},
    headers=places_headers
)
places = resp.json().get("places", [])

print("Places with coordinates:")
for p in places[:5]:
    name = p["displayName"]["text"]
    loc = p["location"]
    print(f"  {name}: ({loc['latitude']:.5f}, {loc['longitude']:.5f})")

Places with coordinates:
  Sharetea - UCI: (33.65056, -117.83812)
  Cha For Tea - University Center: (33.65072, -117.83895)
  Pink Pig: (33.64977, -117.83902)
  TP TEA – Irvine Main: (33.68155, -117.84365)
  Ever After Tea Room & Eatery: (33.66371, -117.82590)


In [9]:
# --- Use lat/lng coordinates in Routes API ---

if len(places) >= 2:
    origin_loc = places[0]["location"]
    dest_loc = places[1]["location"]

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "routes.distanceMeters,routes.duration,routes.polyline.encodedPolyline"
    }

    body = {
        "origin": {
            "location": {
                "latLng": {
                    "latitude": origin_loc["latitude"],
                    "longitude": origin_loc["longitude"]
                }
            }
        },
        "destination": {
            "location": {
                "latLng": {
                    "latitude": dest_loc["latitude"],
                    "longitude": dest_loc["longitude"]
                }
            }
        },
        "travelMode": "DRIVE"
    }

    resp = requests.post(ROUTES_URL, json=body, headers=headers)
    result = resp.json()
    r = result["routes"][0]

    print(f"{places[0]['displayName']['text']} -> {places[1]['displayName']['text']}")
    print(f"Distance: {r['distanceMeters'] / 1000:.1f} km")
    print(f"Duration: {int(r['duration'].rstrip('s')) / 60:.0f} min")
    print(f"Polyline available: {'polyline' in r}")

Sharetea - UCI -> Cha For Tea - University Center
Distance: 0.2 km
Duration: 2 min
Polyline available: True


## 7. Putting It All Together: Itinerary Route Builder

This is what your backend endpoint would look like — take a list of places,
compute the full route with polyline, and return everything the frontend needs.

In [10]:
def compute_itinerary_route(place_list: list[dict], travel_mode: str = "DRIVE") -> dict:
    """
    Given a list of places (each with 'name', 'address', 'latitude', 'longitude'),
    compute the full route and return data ready for the frontend map.
    
    Returns:
        {
            "totalDistanceKm": float,
            "totalDurationMin": float,
            "polyline": str (encoded),
            "legs": [
                { "from": str, "to": str, "distanceKm": float, "durationMin": float }
            ],
            "stops": [
                { "name": str, "latitude": float, "longitude": float }
            ]
        }
    """
    if len(place_list) < 2:
        return {"error": "Need at least 2 places"}

    def make_waypoint(place):
        return {
            "location": {
                "latLng": {
                    "latitude": place["latitude"],
                    "longitude": place["longitude"]
                }
            }
        }

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "routes.distanceMeters,"
            "routes.duration,"
            "routes.polyline.encodedPolyline,"
            "routes.legs.distanceMeters,"
            "routes.legs.duration"
        )
    }

    body = {
        "origin": make_waypoint(place_list[0]),
        "destination": make_waypoint(place_list[-1]),
        "travelMode": travel_mode
    }

    # Add intermediates if more than 2 stops
    if len(place_list) > 2:
        body["intermediates"] = [make_waypoint(p) for p in place_list[1:-1]]

    resp = requests.post(ROUTES_URL, json=body, headers=headers)
    data = resp.json()

    if "routes" not in data or not data["routes"]:
        return {"error": "No route found", "details": data}

    route = data["routes"][0]

    legs = []
    for i, leg in enumerate(route.get("legs", [])):
        legs.append({
            "from": place_list[i]["name"],
            "to": place_list[i + 1]["name"],
            "distanceKm": round(leg["distanceMeters"] / 1000, 1),
            "durationMin": round(int(leg["duration"].rstrip("s")) / 60, 1)
        })

    return {
        "totalDistanceKm": round(route["distanceMeters"] / 1000, 1),
        "totalDurationMin": round(int(route["duration"].rstrip("s")) / 60, 1),
        "polyline": route["polyline"]["encodedPolyline"],
        "legs": legs,
        "stops": [
            {"name": p["name"], "latitude": p["latitude"], "longitude": p["longitude"]}
            for p in place_list
        ]
    }

print("Function defined. Run the next cell to test it.")

Function defined. Run the next cell to test it.


In [11]:
# --- Test with real places from the Places API ---

# Build a list of places with coordinates from our earlier search
test_places = []
for p in places[:4]:  # first 4 boba spots
    test_places.append({
        "name": p["displayName"]["text"],
        "address": p["formattedAddress"],
        "latitude": p["location"]["latitude"],
        "longitude": p["location"]["longitude"]
    })

print("Stops:")
for i, p in enumerate(test_places):
    print(f"  {i+1}. {p['name']}")

result = compute_itinerary_route(test_places, travel_mode="DRIVE")

print(f"\nTotal: {result['totalDistanceKm']} km, {result['totalDurationMin']} min")
print(f"\nLegs:")
for leg in result["legs"]:
    print(f"  {leg['from']} -> {leg['to']}: {leg['distanceKm']} km, {leg['durationMin']} min")
print(f"\nPolyline length: {len(result['polyline'])} chars")

Stops:
  1. Sharetea - UCI
  2. Cha For Tea - University Center
  3. Pink Pig
  4. TP TEA – Irvine Main

Total: 6.0 km, 15.1 min

Legs:
  Sharetea - UCI -> Cha For Tea - University Center: 0.2 km, 1.9 min
  Cha For Tea - University Center -> Pink Pig: 0.1 km, 1.6 min
  Pink Pig -> TP TEA – Irvine Main: 5.6 km, 11.7 min

Polyline length: 262 chars


## 8. What the Frontend Needs

The response from `compute_itinerary_route` is everything the frontend map needs:

```json
{
  "totalDistanceKm": 12.3,
  "totalDurationMin": 22,
  "polyline": "encoded_string_here...",
  "legs": [
    { "from": "Place A", "to": "Place B", "distanceKm": 3.1, "durationMin": 7 }
  ],
  "stops": [
    { "name": "Place A", "latitude": 33.64, "longitude": -117.84 }
  ]
}
```

On the frontend (`react-native-maps`):
1. Decode the polyline -> draw a `<Polyline>` on the map
2. Use `stops` array -> place numbered `<Marker>` pins
3. Use `legs` array -> show travel time cards between stops
4. Use `totalDistanceKm` / `totalDurationMin` -> show trip summary

## Summary: Field Mask Cheat Sheet

| What you need | Field mask |
|---------------|------------|
| Distance only | `routes.distanceMeters` |
| Duration only | `routes.duration` |
| Route polyline | `routes.polyline.encodedPolyline` |
| Leg details | `routes.legs.distanceMeters,routes.legs.duration` |
| Leg polylines | `routes.legs.polyline.encodedPolyline` |
| Step-by-step | `routes.legs.steps.polyline.encodedPolyline` |
| Everything for a map | All of the above combined |

**Billing note:** Polylines and legs are included in the base route computation cost. Traffic-aware polylines cost extra.

In [12]:
import folium

# 1. Create a map centered on the first stop
# We use the coordinates from your 'result' dictionary
start_location = [result['stops'][0]['latitude'], result['stops'][0]['longitude']]
m = folium.Map(location=start_location, zoom_start=14)

# 2. Decode the polyline string into actual coordinates
# We reuse the 'decode_polyline' function you defined in Step 3
route_coordinates = decode_polyline(result['polyline'])

# 3. Draw the route line (Polyline)
folium.PolyLine(
    route_coordinates,
    color="blue",
    weight=5,
    opacity=0.8
).add_to(m)

# 4. Add markers for each stop
for i, stop in enumerate(result['stops']):
    # Add a marker with a popup showing the stop name
    folium.Marker(
        location=[stop['latitude'], stop['longitude']],
        popup=f"{i+1}. {stop['name']}",
        tooltip=stop['name'],
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(m)

m.save("my_route_map.html")
# 5. Display the map